# Launch & Record — 팀원용 단일 노트북

이 노트북 한 개로 **카메라 + 모터 + bag 레코더**를 켜고 두 카메라 화면을 보면서 녹화한다.

**전제 — 한 번만 빌드** (터미널에서):
```bash
cd /home/hyunseo/Personal_Research/AUE4040/final_project/ros2_ws
colcon build --symlink-install
source install/setup.bash
```
그 후 이 노트북을 같은 셸 환경에서 `jupyter notebook`으로 실행 (`ros2` 명령이 PATH에 있어야 함).

**실행 순서:**
1. Config + ROS init
2. **Launch** — `ros2 launch rover_recorder record.launch.py` 실행 (카메라+모터+bag 노드)
3. **Preview** — 두 카메라 라이브 화면
4. **Teleop 안내** — 별도 SSH 터미널에서 `ros2 run rover_teleop teleop_node` 실행
5. **Stop** — 끝낼 때

**왜 teleop만 따로 터미널?** cbreak 키 입력은 자기 TTY를 점유해야 동작. 노트북 subprocess로 띄우면 키가 안 먹힘. 카메라/모터/bag은 노트북에서 띄워도 무관.

## 1. Config + ROS init

In [ ]:
SESSION_NAME = 'session01'
OUT_ROOT     = '/home/hyunseo/rover_data'
FPS          = 15
DRY_RUN      = False    # True면 모터 안 움직임 (UART 안 씀)

import os, sys, time, signal, subprocess, threading, shutil
from pathlib import Path

import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display

if shutil.which('ros2') is None:
    raise RuntimeError(
        'ros2 not on PATH. 터미널에서 `source install/setup.bash` 후 jupyter 실행하세요.'
    )

import rclpy
from rclpy.node import Node
from sensor_msgs.msg import CompressedImage
from std_msgs.msg import Bool

if not rclpy.ok():
    rclpy.init()
print('ros2:', shutil.which('ros2'))
print('rclpy ok')

## 2. Launch — 카메라 + 모터 + bag 레코더

`ros2 launch rover_recorder record.launch.py`를 subprocess로 띄움. 로그는 아래 출력 영역에 흘러나옴.

**이미 떠 있는 ros2 프로세스가 있으면 충돌**할 수 있으니, 처음 실행 시 다른 노드는 다 죽이고 시작.

In [ ]:
launch_proc = {'p': None}
log_output = widgets.Output(layout=widgets.Layout(
    border='1px solid gray', height='250px', overflow='auto'))
display(log_output)

def _pump(p):
    for line in iter(p.stdout.readline, b''):
        with log_output:
            print(line.decode(errors='replace').rstrip())

def start_launch():
    if launch_proc['p'] is not None and launch_proc['p'].poll() is None:
        print('already running'); return
    cmd = [
        'ros2', 'launch', 'rover_recorder', 'record.launch.py',
        f'session_name:={SESSION_NAME}',
        f'out_root:={OUT_ROOT}',
        f'fps:={FPS}',
        f'dry_run:={str(DRY_RUN).lower()}',
    ]
    with log_output:
        print('LAUNCH:', ' '.join(cmd))
    p = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        preexec_fn=os.setsid,
    )
    launch_proc['p'] = p
    threading.Thread(target=_pump, args=(p,), daemon=True).start()

def stop_launch():
    p = launch_proc['p']
    if p is None or p.poll() is not None:
        print('not running'); launch_proc['p'] = None; return
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGINT)
        p.wait(timeout=8)
    except Exception:
        try: p.kill()
        except Exception: pass
    launch_proc['p'] = None
    print('launch stopped')

btn_start = widgets.Button(description='▶ START launch', button_style='success')
btn_stop  = widgets.Button(description='■ STOP launch',  button_style='danger')
btn_start.on_click(lambda _: start_launch())
btn_stop.on_click(lambda _: stop_launch())
display(widgets.HBox([btn_start, btn_stop]))

## 3. Preview — 두 카메라 라이브 화면

`/bev_image/compressed`, `/front_image/compressed` 구독해서 jpeg widget으로 표시. `age`가 1초 이상으로 늘어나면 카메라 publish가 끊긴 것.

In [ ]:
PREVIEW_W = 480

class Viewer(Node):
    def __init__(self):
        super().__init__('rover_viewer_nb')
        self.latest = {'bev': None, 'front': None}
        self.last_t = {'bev': 0.0, 'front': 0.0}
        self.create_subscription(CompressedImage, '/bev_image/compressed',
                                 lambda m: self._on(m, 'bev'),   10)
        self.create_subscription(CompressedImage, '/front_image/compressed',
                                 lambda m: self._on(m, 'front'), 10)

    def _on(self, msg, key):
        arr = np.frombuffer(msg.data, np.uint8)
        bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if bgr is not None:
            self.latest[key] = bgr
            self.last_t[key] = time.time()

viewer = Viewer()
stop_flag = {'stop': False}

def _spin():
    while not stop_flag['stop'] and rclpy.ok():
        rclpy.spin_once(viewer, timeout_sec=0.05)
threading.Thread(target=_spin, daemon=True).start()

img_bev   = widgets.Image(format='jpeg', width=PREVIEW_W)
img_front = widgets.Image(format='jpeg', width=PREVIEW_W)
status    = widgets.HTML()
display(widgets.VBox([
    widgets.HBox([widgets.VBox([widgets.Label('BEV  (/bev_image/compressed)'),   img_bev]),
                  widgets.VBox([widgets.Label('FRONT (/front_image/compressed)'), img_front])]),
    status,
]))

def _preview():
    while not stop_flag['stop']:
        now = time.time()
        for key, w in [('bev', img_bev), ('front', img_front)]:
            bgr = viewer.latest[key]
            if bgr is None:
                continue
            h, wpx = bgr.shape[:2]
            scale = PREVIEW_W / wpx
            small = cv2.resize(bgr, (PREVIEW_W, int(h * scale)))
            ok, jpg = cv2.imencode('.jpg', small, [cv2.IMWRITE_JPEG_QUALITY, 70])
            if ok:
                w.value = jpg.tobytes()
        age_b = now - viewer.last_t['bev']   if viewer.last_t['bev']   else float('inf')
        age_f = now - viewer.last_t['front'] if viewer.last_t['front'] else float('inf')
        status.value = f'<b>age</b> bev={age_b:.2f}s  front={age_f:.2f}s'
        time.sleep(0.1)
threading.Thread(target=_preview, daemon=True).start()
print('preview running')

## 4. Teleop 안내 — 별도 SSH 터미널에서 실행

이 노트북과 **다른 SSH 터미널**을 열고:
```bash
cd /home/hyunseo/Personal_Research/AUE4040/final_project/ros2_ws
source install/setup.bash
ros2 run rover_teleop teleop_node
```

**키 매핑** (teleop 터미널에서):
- `a` / `d` : turn_level −1 / +1 (−5..+5)
- `space` : 정지
- `g` : drive on/off (모터 송신 토글)
- `r` : 녹화 on/off (bag 시작/종료)
- `1` `2` `3` `4` : segment 라벨 common/left/right/pause
- `q` / `ESC` : 종료

녹화 toggle 상태는 아래 indicator로 확인 가능.

In [ ]:
rec_state = {'on': False}

class RecIndicator(Node):
    def __init__(self):
        super().__init__('rover_rec_indicator_nb')
        self.create_subscription(Bool, '/record_enable',
                                 lambda m: rec_state.update(on=bool(m.data)), 10)

rec_node = RecIndicator()
def _spin_rec():
    while not stop_flag['stop'] and rclpy.ok():
        rclpy.spin_once(rec_node, timeout_sec=0.1)
threading.Thread(target=_spin_rec, daemon=True).start()

rec_label = widgets.HTML()
display(rec_label)
def _rec_loop():
    while not stop_flag['stop']:
        color = 'red' if rec_state['on'] else 'gray'
        word  = 'RECORDING' if rec_state['on'] else 'idle'
        rec_label.value = (f"<span style='color:{color};font-weight:bold;font-size:18px'>"
                           f"● {word}</span>")
        time.sleep(0.2)
threading.Thread(target=_rec_loop, daemon=True).start()
print('rec indicator running')

## 5. Stop — 끝낼 때

**순서 중요**: teleop 터미널에서 먼저 `q`로 종료해서 모터 정지 시킨 다음, 아래 셀로 launch 종료.

In [ ]:
stop_flag['stop'] = True
time.sleep(0.3)
stop_launch()
try: viewer.destroy_node()
except Exception: pass
try: rec_node.destroy_node()
except Exception: pass
if rclpy.ok():
    rclpy.shutdown()
print('done')